# Fine-Tuning PhoBERT on Stratified 70/15/15 Merged Emotion Dataset

This notebook loads pre-split **Train (70% - 7,088 samples)**, **Validation (15% - 1,519 samples)**, and **Test (15% - 1,520 samples)** datasets generated from the merged VSMEC + GoEmotions corpus. It performs fine-tuning with Early Stopping and evaluates directly on the test set.

In [ ]:
# Step 1: Install required libraries
!pip install -q transformers datasets torch accelerate scikit-learn matplotlib


In [ ]:
# Step 2: Load train, val, test datasets
import json
import numpy as np

with open("phobert_train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open("phobert_val.json", "r", encoding="utf-8") as f:
    val_data = json.load(f)
with open("phobert_test.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Train samples: {len(train_data)}")
print(f"Val samples  : {len(val_data)}")
print(f"Test samples : {len(test_data)}")


In [ ]:
# Step 3: Tokenization & HuggingFace Dataset Conversion
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import classification_report, accuracy_score, f1_score
import torch

model_name = "vinai/phobert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_ds = Dataset.from_dict({"text": [x["text"] for x in train_data], "label": [x["label"] for x in train_data]}).map(tokenize_fn, batched=True)
val_ds = Dataset.from_dict({"text": [x["text"] for x in val_data], "label": [x["label"] for x in val_data]}).map(tokenize_fn, batched=True)
test_ds = Dataset.from_dict({"text": [x["text"] for x in test_data], "label": [x["label"] for x in test_data]}).map(tokenize_fn, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=7)


In [ ]:
# Step 4: Fine-Tuning Setup with Early Stopping on Validation Set (Max Epochs: 5)
training_args = TrainingArguments(
    output_dir="./results_phobert",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True,
    logging_steps=50
)

def compute_metrics(eval_pred):
    logits, l_labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(l_labels, preds)
    macro_f1 = f1_score(l_labels, preds, average="macro")
    return {"accuracy": acc, "f1": macro_f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

print("Starting PhoBERT Fine-Tuning...")
trainer.train()

# Save Best Model
model.save_pretrained("./phobert_emotion_final")
tokenizer.save_pretrained("./phobert_emotion_final")
print("Best model saved to ./phobert_emotion_final")


In [ ]:
# Step 5: Final Evaluation on Independent Test Set (15% - 1,520 samples)
print("\n==================================================")
print("   EVALUATING BEST MODEL ON HELD-OUT TEST SET")
print("==================================================")

test_results = trainer.predict(test_ds)
test_preds = np.argmax(test_results.predictions, axis=1)
test_labels = [x["label"] for x in test_data]
label_names = ["Enjoyment", "Sadness", "Disgust", "Anger", "Fear", "Surprise", "Other"]

report_str = classification_report(test_labels, test_preds, target_names=label_names, digits=4)
print(report_str)

# Zip final model for export
!zip -r phobert_emotion_final.zip ./phobert_emotion_final


In [ ]:
# Step 6: Plot Training & Validation Learning Curves for Performance Verification
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))

history = trainer.state.log_history
epochs = []
val_f1s = []
train_losses = []
val_losses = []

for log in history:
    if "eval_f1" in log:
        epochs.append(log["epoch"])
        val_f1s.append(log["eval_f1"] * 100)
        val_losses.append(log["eval_loss"])
    elif "loss" in log:
        train_losses.append(log["loss"])

plt.subplot(1, 2, 1)
if len(epochs) > 0 and len(val_f1s) == len(epochs):
    plt.plot(epochs, val_f1s, "o-", color="tab:blue", linewidth=2, label="Validation Macro F1")
plt.title("Validation Macro F1-Score per Epoch", fontsize=12, fontweight="bold")
plt.xlabel("Epoch", fontsize=10)
plt.ylabel("Macro F1 (%)", fontsize=10)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.subplot(1, 2, 2)
if len(epochs) > 0 and len(val_losses) == len(epochs):
    plt.plot(epochs, val_losses, "o-", color="tab:red", linewidth=2, label="Validation Loss")
plt.title("Training Convergence & Validation Loss", fontsize=12, fontweight="bold")
plt.xlabel("Epoch", fontsize=10)
plt.ylabel("Loss", fontsize=10)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig("training_performance_curves.png", dpi=300)
plt.show()
print("Learning curves plot saved successfully to training_performance_curves.png!")